# Model Architecture Comparison

`Regularization_v2.ipynb`의 학습 스타일을 기준으로, `src/models`에 있는 모델 architecture별 validation 성능을 같은 조건에서 비교합니다.

- 같은 train/dev split 사용
- 같은 augmentation, optimizer, cosine scheduler, EMA, early stopping 사용
- distillation은 제외합니다. 원본 노트북의 distillation은 `FlexibleMultiViewFeatureFusion(return_feat=True)` 전용이라, architecture 비교에서는 공정하게 `model(views) -> logits`만 사용합니다.
- shared-weight 모델은 같은 backbone 설정을 사용하고, dual-encoder 모델만 front/top backbone을 각각 따로 생성합니다.


In [ ]:
from pathlib import Path
import sys

def find_project_root():
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        if (base / 'src').is_dir() and (base / 'data').is_dir():
            return base
    raise RuntimeError('Could not find project root.')

ROOT = find_project_root()
TOOLS_DIR = ROOT / 'tools'
if str(TOOLS_DIR) not in sys.path:
    sys.path.insert(0, str(TOOLS_DIR))

from compare_model_architectures import DEFAULT_CFG, get_model_specs, run_comparison

print('ROOT:', ROOT)
print('Available models:')
for spec in get_model_specs():
    print(f"- {spec.name}: {spec.display_name}")


## Configuration

처음 확인할 때는 `EPOCHS`, `MAX_TRAIN_SAMPLES`, `MAX_VAL_SAMPLES`를 작게 잡아 smoke test를 한 뒤 전체 학습으로 돌리는 것을 권장합니다. Windows 로컬 Jupyter에서는 `NUM_WORKERS=0`이 가장 안정적입니다.

In [ ]:
CFG = dict(DEFAULT_CFG)

# Regularization_v2.ipynb와 같은 기본값을 유지하되, 필요하면 여기서 덮어쓰세요.
CFG.update({
    'EPOCHS': 100,
    'BATCH_SIZE': 64,
    'LEARNING_RATE': 3e-4,
    'MIN_LR': 1e-6,
    'WEIGHT_DECAY': 1e-4,
    'EMA_DECAY': 0.999,
    'EMA_USE_FOR_EVAL': True,
    'PATIENCE': 10,
    'BACKBONE_NAME': 'efficientnetv2_rw_s',
    'PRETRAINED': True,
    'CHECKERBOARD_TOP_NORMALIZE': True,
    'VIDEO_AUG_ENABLE': True,
    'RUN_MODELS': [
        'feature_fusion',
        'cross_attention',
        'bidirectional_cross_attention',
        'dual_encoder_bidirectional_cross_attention',
    ],
})

# Smoke test example:
# CFG.update({'EPOCHS': 2, 'MAX_TRAIN_SAMPLES': 128, 'MAX_VAL_SAMPLES': 64, 'VIDEO_AUG_ENABLE': False})

CFG


## Run Comparison

In [ ]:
results_df, histories = run_comparison(CFG)
results_df


## Summary Plot

In [ ]:
import matplotlib.pyplot as plt

plot_df = results_df.sort_values('best_val_logloss', ascending=True)
fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(plot_df['display_name'], plot_df['best_val_logloss'])
ax.invert_yaxis()
ax.set_xlabel('Best validation log-loss')
ax.set_title('Architecture comparison')
for i, value in enumerate(plot_df['best_val_logloss']):
    ax.text(value, i, f' {value:.5f}', va='center')
plt.tight_layout()
plt.show()


## Training Curves

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.concat(histories.values(), ignore_index=True)

fig, ax = plt.subplots(figsize=(10, 5))
for name, group in history_df.groupby('display_name'):
    ax.plot(group['epoch'], group['val_logloss'], marker='o', linewidth=1.5, label=name)
ax.set_xlabel('Epoch')
ax.set_ylabel('Validation log-loss')
ax.set_title('Validation curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

history_df.tail()
